# Answer generation (F1.6) — grounded, cited, refuses when unsure

The full Phase 1 RAG loop: retrieve chunks -> feed to Claude under a governance prompt -> grounded answer with citations, or `NOT FOUND`.

**Setup:** put your key in `.env` as `ANTHROPIC_API_KEY=...`. Each answerable question makes one small Claude call; out-of-corpus questions refuse for free (no call).

**How to run:** VS Code with the `.venv` kernel, or `uv run jupyter lab`.

In [ ]:
from policy_copilot.agent import answer
from policy_copilot.index import load_index

index, chunks = load_index()


def ask(question):
    res = answer(question, index, chunks)
    print("Q:", question)
    print("refused:", res.refused)
    print("answer :", res.text)
    print("cited  :", res.citations)
    return res

## 1. An answerable question (in corpus)

In [ ]:
res = ask("What was AMD's total net revenue in fiscal 2022?")

## 2. Provenance — which chunks backed the answer

Every answer can be traced to the exact excerpts the model saw.

In [ ]:
for hit in res.hits:
    print(f"{hit.score:.3f}  [{hit.chunk.chunk_id}]  {hit.chunk.heading[:40]}")

## 3. Out of corpus -> refusal (no Claude call)

The top retrieval score is below threshold, so it refuses before spending a token.

In [ ]:
_ = ask("What is the capital of France?")

## 4. Try your own

Ask about Boeing, PepsiCo, 3M, American Express... or something the filings can't answer.

In [ ]:
_ = ask("How much did Boeing spend on research and development in 2022?")